# USA Housing EDA

Exploratory data analysis for the ingested housing dataset used by the pipeline.

Dataset path: `data/raw/usa-housing-dataset/USA Housing Dataset.csv`

## Goals

- verify the schema and row counts
- inspect nulls, duplicates, and type expectations
- surface data quality issues before transformation and training
- translate EDA findings into validation rules

In [9]:
from pathlib import Path
print(Path.cwd())


/Users/pranavchourasiya/pranav/house-price-ml/notebooks


In [10]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path
from statistics import mean

DATA_PATH = Path('../data/raw/usa-housing-dataset/USA Housing Dataset.csv').resolve()
assert DATA_PATH.exists(), f'Missing dataset file: {DATA_PATH}'

with DATA_PATH.open('r', encoding='utf-8', newline='') as file_obj:
    reader = csv.DictReader(file_obj)
    rows = list(reader)
    columns = reader.fieldnames or []

len(rows), columns

(4140,
 ['date',
  'price',
  'bedrooms',
  'bathrooms',
  'sqft_living',
  'sqft_lot',
  'floors',
  'waterfront',
  'view',
  'condition',
  'sqft_above',
  'sqft_basement',
  'yr_built',
  'yr_renovated',
  'street',
  'city',
  'statezip',
  'country'])

In [11]:
rows[:2]

[{'date': '2014-05-09 00:00:00',
  'price': '376000.0',
  'bedrooms': '3.0',
  'bathrooms': '2.0',
  'sqft_living': '1340',
  'sqft_lot': '1384',
  'floors': '3.0',
  'waterfront': '0',
  'view': '0',
  'condition': '3',
  'sqft_above': '1340',
  'sqft_basement': '0',
  'yr_built': '2008',
  'yr_renovated': '0',
  'street': '9245-9249 Fremont Ave N',
  'city': 'Seattle',
  'statezip': 'WA 98103',
  'country': 'USA'},
 {'date': '2014-05-09 00:00:00',
  'price': '800000.0',
  'bedrooms': '4.0',
  'bathrooms': '3.25',
  'sqft_living': '3540',
  'sqft_lot': '159430',
  'floors': '2.0',
  'waterfront': '0',
  'view': '0',
  'condition': '3',
  'sqft_above': '3540',
  'sqft_basement': '0',
  'yr_built': '2007',
  'yr_renovated': '0',
  'street': '33001 NE 24th St',
  'city': 'Carnation',
  'statezip': 'WA 98014',
  'country': 'USA'}]

In [12]:
missing_counts = {column: sum(1 for row in rows if not row[column].strip()) for column in columns}
duplicate_rows = len(rows) - len({tuple((column, row[column]) for column in columns) for row in rows})
missing_counts, duplicate_rows

({'date': 0,
  'price': 0,
  'bedrooms': 0,
  'bathrooms': 0,
  'sqft_living': 0,
  'sqft_lot': 0,
  'floors': 0,
  'waterfront': 0,
  'view': 0,
  'condition': 0,
  'sqft_above': 0,
  'sqft_basement': 0,
  'yr_built': 0,
  'yr_renovated': 0,
  'street': 0,
  'city': 0,
  'statezip': 0,
  'country': 0},
 0)

In [13]:
numeric_columns = [
    'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated'
]

numeric_summary = {}
for column in numeric_columns:
    values = [float(row[column]) for row in rows]
    numeric_summary[column] = {
        'min': min(values),
        'mean': round(mean(values), 2),
        'max': max(values),
    }

numeric_summary

{'price': {'min': 0.0, 'mean': 553062.88, 'max': 26590000.0},
 'bedrooms': {'min': 0.0, 'mean': 3.4, 'max': 8.0},
 'bathrooms': {'min': 0.0, 'mean': 2.16, 'max': 6.75},
 'sqft_living': {'min': 370.0, 'mean': 2143.64, 'max': 10040.0},
 'sqft_lot': {'min': 638.0, 'mean': 14697.64, 'max': 1074218.0},
 'floors': {'min': 1.0, 'mean': 1.51, 'max': 3.5},
 'waterfront': {'min': 0.0, 'mean': 0.01, 'max': 1.0},
 'view': {'min': 0.0, 'mean': 0.25, 'max': 4.0},
 'condition': {'min': 1.0, 'mean': 3.45, 'max': 5.0},
 'sqft_above': {'min': 370.0, 'mean': 1831.35, 'max': 8020.0},
 'sqft_basement': {'min': 0.0, 'mean': 312.29, 'max': 4820.0},
 'yr_built': {'min': 1900.0, 'mean': 1970.81, 'max': 2014.0},
 'yr_renovated': {'min': 0.0, 'mean': 808.37, 'max': 2014.0}}

In [14]:
zero_or_negative_price_rows = [row for row in rows if float(row['price']) <= 0]
len(zero_or_negative_price_rows), zero_or_negative_price_rows[:3]

(49,
 [{'date': '2014-05-05 00:00:00',
   'price': '0.0',
   'bedrooms': '3.0',
   'bathrooms': '1.75',
   'sqft_living': '1490',
   'sqft_lot': '10125',
   'floors': '1.0',
   'waterfront': '0',
   'view': '0',
   'condition': '4',
   'sqft_above': '1490',
   'sqft_basement': '0',
   'yr_built': '1962',
   'yr_renovated': '0',
   'street': '3911 S 328th St',
   'city': 'Federal Way',
   'statezip': 'WA 98001',
   'country': 'USA'},
  {'date': '2014-05-05 00:00:00',
   'price': '0.0',
   'bedrooms': '4.0',
   'bathrooms': '2.75',
   'sqft_living': '2600',
   'sqft_lot': '5390',
   'floors': '1.0',
   'waterfront': '0',
   'view': '0',
   'condition': '4',
   'sqft_above': '1300',
   'sqft_basement': '1300',
   'yr_built': '1960',
   'yr_renovated': '2001',
   'street': '2120 31st Ave W',
   'city': 'Seattle',
   'statezip': 'WA 98199',
   'country': 'USA'},
  {'date': '2014-05-05 00:00:00',
   'price': '0.0',
   'bedrooms': '6.0',
   'bathrooms': '2.75',
   'sqft_living': '3200',
   's

In [15]:
country_counts = Counter(row['country'] for row in rows)
city_counts = Counter(row['city'] for row in rows).most_common(10)
state_prefix_counts = Counter(row['statezip'].split()[0] for row in rows).most_common(10)
country_counts, city_counts, state_prefix_counts

(Counter({'USA': 4140}),
 [('Seattle', 1415),
  ('Renton', 261),
  ('Bellevue', 260),
  ('Redmond', 209),
  ('Kent', 167),
  ('Kirkland', 166),
  ('Issaquah', 162),
  ('Auburn', 162),
  ('Sammamish', 158),
  ('Federal Way', 131)],
 [('WA', 4140)])

## EDA Findings To Feed The Pipeline

- The dataset has `4140` rows and `18` columns.
- No missing values were found in the current download.
- No exact duplicate rows were found.
- `country` is constant as `USA`; `statezip` values are all prefixed with `WA`.
- `price` contains `49` non-positive values, which should be treated as a data quality warning and handled before model training.
- `sqft_living`, `sqft_lot`, `sqft_above`, and `yr_built` are expected to remain strictly positive.
- The `date` column follows the `%Y-%m-%d %H:%M:%S` format and should be validated as such.